### Full experiment replicating FinGAN implementation

reference: https://github.com/milenavuletic/Fin-GAN/blob/main/README.md

In [16]:
#Importing packages
import os
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import seaborn as sns
import random
import torch.nn as nn
import torch.optim as optim
import numpy.random as rnd
from tqdm import tqdm
from IPython.display import clear_output
import contextlib, io
from itertools import product
from scipy.stats import ttest_rel
from pmdarima import auto_arima
from sklearn.metrics import mean_squared_error
from joblib import Parallel, delayed

In [11]:
df = pd.read_csv("datasets/stocks-etfs-list.csv")
df.columns

df['Sector'].unique()

array(['Industrials', 'Health Care', 'Information Technology',
       'Utilities', 'Financials', 'Materials', 'Consumer Discretionary',
       'Real Estate', 'Communication Services', 'Consumer Staples',
       'Energy'], dtype=object)

In [12]:
df[['Sector', 'SectorTicker']].drop_duplicates()



,Sector,SectorTicker
0,Industrials,XLI
2,Health Care,XLV
4,Information Technology,XLK
7,Utilities,XLU
8,Financials,XLF
10,Materials,XLB
11,Consumer Discretionary,XLY
14,Real Estate,XLRE
19,Communication Services,XLC
21,Consumer Staples,XLP


In [ ]:
# Ignore XLC and XLRE
# we have 9 sectors, in the Data preprocessing we chose 5: 
# 1. Information Technology 
# 2. Consumer Discretionary
# 3. Financials                    
# 4. Health Care                  
# 5. Energy   

In [5]:
# Personalised functions

# Excess returns function
def excessreturns(dataloc, stock, etf, plotcheck = False):
    """
    function to get a time series of alternating close and open
    etf-excess log returns for a given stock
    all prices are adjusted for stock events
    input: location of datasets, stock ticker, etf ticker
    output: time series of etf excess log returns
    optional: plot sanity check
    """
    s_df = pd.read_csv(dataloc+stock+".csv")
    e_df = pd.read_csv(dataloc+etf+".csv")
    dates_dt = pd.to_datetime(s_df['date'])
    d1 = pd.to_datetime("2025-03-01")
    smp = (dates_dt < d1)
    s_df = s_df[smp]
    dates_dt = pd.to_datetime(s_df['date'])
    e_df = e_df[smp]
    s_logclose = np.log(s_df['AdjClose'])
    e_logclose = np.log(e_df['AdjClose'])
    s_logopen = np.log(s_df['AdjOpen'])
    e_logopen = np.log(e_df['AdjOpen'])
    s_log = np.zeros(2*len(s_logclose))
    e_log = np.zeros(2*len(s_logclose))
    for i in range(len(s_logclose)):
        s_log[2 * i] = s_logopen[i]   # open
        s_log[2 * i + 1] = s_logclose[i]   # close
        e_log[2 * i] = e_logopen[i]
        e_log[2 * i + 1] = e_logclose[i]
    s_ret = np.diff(s_log)  # stock returns (alternating overnight/intraday)
    e_ret = np.diff(e_log)   # ETF returns (same)
    # clip extremes returns
    s_ret[s_ret > 0.15] = 0.15
    s_ret[s_ret < -0.15] = -0.15
    e_ret[e_ret > 0.15] = 0.15
    e_ret[e_ret < -0.15] = -0.15

    # compute excess returns
    # subtracts ETF returns from stock returns -> gives alpha style signal (stock's behavior not explained by sector ETF)
    excessret = s_ret - e_ret
    dates_dt = pd.to_datetime(s_df['date']) # daily dates from stock file

    # shift because np.diff loses 1 element
    dates_double = np.repeat(dates_dt.values, 2)[1:]  # ADDED

    # Optional print to verify alignment
    print(f"[excessreturns] Length excess_ret: {len(excessret)}, Length dates_double: {len(dates_double)}") # ADDED

    print(f"[excessreturns] Sample s_ret:\n{s_ret[:5]}")
    print(f"[excessreturns] Sample e_ret:\n{e_ret[:5]}")
    print(f"[excessreturns] Sample excess_ret:\n{excessret[:5]}")
    
    #plots show price evolution, stock vs ETF returns, excess return over time
    if plotcheck:
        plt.figure(stock+" price")
        plt.title(stock+" price")
        plt.plot(dates_dt,s_df['AdjClose'])
        plt.xlabel("date")
        plt.ylabel("price in USD")
        plt.show()
        plt.figure("Returns "+stock)
        plt.title("Returns "+stock)
        plt.plot(dates_double, s_ret, alpha=0.7, label='stock') 
        plt.plot(dates_double, e_ret, alpha=0.7, label='etf')
        plt.plot(dates_double, excessret, alpha=0.7, label='excess return')
    
        plt.legend()
        plt.show()

    return excessret, pd.to_datetime(dates_double) 


def ETF_find(etflistloc, stock):
    """
    Reads ETF-sector mapping and finds the ETF ticker for a given stock.
    """
    print(f"[ETF_find] Looking for ETF match for stock: {stock}")
    data = pd.read_csv(etflistloc)

    if stock not in data['ticker_x'].values:
        print(f"[ETF_find] ❌ Stock {stock} not found in ETF list!")
        return None

    etf = np.array(data['ticker_y'][data['ticker_x'] == stock])[0]
    print(f"[ETF_find] ✅ Found ETF {etf} for stock {stock}")
    return etf


# Simplified split version stocks


def split_from_returns(excess_returns, dates_dt, tr=0.8, vl=0.1, h=1, l=10, pred=1, plotcheck=False):
    """
    Splits precomputed excess returns into train, val, and test sets using a sliding window.

    Parameters:
        excess_returns (np.array): 1D array of excess returns
        dates_dt (np.array): Matching array of datetime values
        tr (float): Training proportion
        vl (float): Validation proportion
        h (int): Step size of sliding window
        l (int): Lookback window
        pred (int): Prediction window
        plotcheck (bool): Whether to plot splits

    Returns:
        train_data, val_data, test_data, dates_dt
    """
    N = len(excess_returns)
    N_tr = int(tr * N)   # num training
    N_vl = int(vl * N)    # num validation
    N_tst = N - N_tr - N_vl  # num test

    train_sr = excess_returns[0:N_tr]
    val_sr = excess_returns[N_tr:N_tr + N_vl]
    test_sr = excess_returns[N_tr + N_vl:]

    def create_split(series, name):
        n_samples = int((len(series) - l - pred) / h) + 1
        print(f"[{name}] Creating {n_samples} samples")
        split_data = np.zeros((n_samples, l + pred))
        for i in range(n_samples):
            split_data[i, :] = series[i * h:i * h + l + pred]
        print(f"[{name}] Sample (first 2 rows):\n{split_data[:2]}")
        return split_data

    train_data = create_split(train_sr, "Train")
    val_data = create_split(val_sr, "Val")
    test_data = create_split(test_sr, "Test")

    if plotcheck:
        plt.figure(figsize=(12, 4))
        plt.plot(dates_dt, excess_returns, label="Excess Returns")
        plt.axvline(x=dates_dt[N_tr], color="red", linestyle='--', label="Train/Val Split")
        plt.axvline(x=dates_dt[N_tr + N_vl], color="green", linestyle='--', label="Val/Test Split")
        plt.title("Split Visualization")
        plt.legend()
        plt.grid(True)
        plt.show()

    return train_data, val_data, test_data, dates_dt

# Simplified split function (after returns) -- ETF

# split_train_val_testraw
from FinGAN import rawreturns

def split_from_rawreturns(stock, dataloc, tr=0.8, vl=0.1, h=1, l=10, pred=1, plotcheck=False):
    """
    Applies rawreturns to ETFs and splits into train/val/test with sliding window.
    Structure matches split_from_returns.
    """
    raw_ret, dates_dt = rawreturns(dataloc, stock, plotcheck)
    N = len(raw_ret)
    N_tr = int(tr * N)
    N_vl = int(vl * N)
    N_tst = N - N_tr - N_vl

    train_sr = raw_ret[0:N_tr]
    val_sr = raw_ret[N_tr:N_tr + N_vl]
    test_sr = raw_ret[N_tr + N_vl:]

    def create_split(series, name):
        n_samples = int((len(series) - l - pred) / h) + 1
        print(f"[{name}] Creating {n_samples} samples")
        split_data = np.zeros((n_samples, l + pred))
        for i in range(n_samples):
            split_data[i, :] = series[i * h:i * h + l + pred]
        print(f"[{name}] Sample (first 2 rows):\n{split_data[:2]}")
        return split_data

    train_data = create_split(train_sr, "Train")
    val_data = create_split(val_sr, "Val")
    test_data = create_split(test_sr, "Test")

    if plotcheck:
        plt.figure(figsize=(12, 4))
        plt.plot(dates_dt, raw_ret, label="Raw Returns")
        plt.axvline(x=dates_dt[N_tr], color="red", linestyle='--', label="Train/Val Split")
        plt.axvline(x=dates_dt[N_tr + N_vl], color="green", linestyle='--', label="Val/Test Split")
        plt.title(f"Split Visualization: {stock}")
        plt.legend()
        plt.grid(True)
        plt.show()

    return train_data, val_data, test_data, dates_dt


# LSTM
class LSTM(nn.Module):
    '''
    Values:
        cond_dim: the dimension of the condition, a scalar
        hidden_dim: the inner dimension, a scalar
    '''
    def __init__(self, noise_dim,cond_dim, hidden_dim,output_dim,mean,std):
        super(LSTM, self).__init__()
        self.input_dim = noise_dim+cond_dim
        self.cond_dim = cond_dim
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim
        self.noise_dim = noise_dim
        #predicting a single value, so the output dimension is 1
        self.mean = mean
        self.std = std

        self.lstm = nn.LSTM(
            input_size=cond_dim,
            hidden_size=self.hidden_dim,   
            num_layers=1,
            dropout=0
        )
        # Linear layer to map LSTM output to prediction
        self.linear = nn.Linear(hidden_dim, output_dim) 

        self.activation = nn.ReLU()
       

    def forward(self, condition,h_0,c_0):
        '''
        Function for completing a forward pass of the generator:adding the noise and the condition separately
        '''
        #x = combine_vectors(noise.to(torch.float),condition.to(torch.float),2)
        condition = (condition-self.mean)/self.std
        out, (h_n, c_n) = self.lstm(condition, (h_0, c_0)) # out: [1, batch_size, hidden_dim]
        #print("[DEBUG] LSTM output before linear:", out.shape)  # should be [1, batch, hidden_dim]
        #out = self.linear(out)  # Map to [1, batch_size, pred]  # ADDED
        #out = out*self.std+self.mean 

        # Reshape for linear: (1, batch_size, hidden_dim) → (batch_size, hidden_dim)
        out = out.squeeze(0)  # [batch_size, hidden_dim]
        out = self.linear(out)  # [batch_size, output_dim]
        out = out.unsqueeze(0)  # [1, batch_size, output_dim]

        #print("[DEBUG] LSTM output after linear:", out.shape)  # should be [1, batch_size, 1]
        
        # Denormalize
        out = out * self.std + self.mean
         
        return out

In [31]:
from FinGAN import combine_vectors
def GradientCheck(ticker, gen, disc, gen_opt, disc_opt, criterion, n_epochs, train_data,batch_size,hid_d, hid_g, z_dim, lr_d = 0.0001, lr_g = 0.0001, h = 1, l = 10, pred = 1, diter =1, tanh_coeff = 100, device = 'cpu', plot = False):
    """
    Gradient norm check
    """
    ntrain = train_data.shape[0]
    nbatches = ntrain//batch_size+1
    BCE_norm = torch.empty(nbatches*n_epochs, device = device)
    PnL_norm = torch.empty(nbatches*n_epochs, device = device)
    MSE_norm = torch.empty(nbatches*n_epochs, device = device)
    SR_norm = torch.empty(nbatches*n_epochs, device = device)
    STD_norm = torch.empty(nbatches*n_epochs, device = device)

    fake_and_condition = False
    real_and_condition = False

    disc_fake_pred = False
    disc_real_pred = False
    totlen = train_data.shape[0]

    #currstep = 0
    #train the discriminator more

    gen.train()

    # mini training process for n_epochs, each with nbatches
    for epoch in tqdm(range(n_epochs)):
        perm = torch.randperm(ntrain)
        train_data = train_data[perm,:]
        #shuffle the dataset for the optimisation to work
        for i in range(nbatches):
            curr_batch_size = batch_size
            if i==(nbatches-1):
                curr_batch_size = totlen-i*batch_size
            h_0d = torch.zeros((1,curr_batch_size,hid_d),device=device,dtype= torch.float)
            c_0d = torch.zeros((1,curr_batch_size,hid_d),device=device,dtype= torch.float)
            h_0g = torch.zeros((1,curr_batch_size,hid_g),device=device,dtype= torch.float)
            c_0g = torch.zeros((1,curr_batch_size,hid_g),device=device,dtype= torch.float)
            # condition is the lookback, real is the future
            condition = train_data[(i*batch_size):(i*batch_size+curr_batch_size),0:l]
            condition = condition.unsqueeze(0)
            real = train_data[(i*batch_size):(i*batch_size+curr_batch_size),l:(l+pred)]
            real = real.unsqueeze(0)

            ### Update discriminator ###
            # Zero out the discriminator gradients
            for j in range(diter):
                disc_opt.zero_grad()
            # Get noise corresponding to the current batch_size
                noise = torch.randn(1,curr_batch_size, z_dim, device=device,dtype=torch.float)
            # Get outputs from the generator
                fake = gen(noise,condition,h_0g,c_0g)
                # fake = fake.unsqueeze(0)
                fake_and_condition = combine_vectors(condition,fake,dim=-1)
                fake_and_condition.to(torch.float)
                real_and_condition = combine_vectors(condition,real,dim=-1)
                disc_fake_pred = disc(fake_and_condition.detach(),h_0d,c_0d)
                disc_real_pred = disc(real_and_condition,h_0d,c_0d)

            #Updating the discriminator

                disc_fake_loss = criterion(disc_fake_pred, torch.zeros_like(disc_fake_pred))
                disc_real_loss = criterion(disc_real_pred, torch.ones_like(disc_real_pred))
                disc_loss = (disc_fake_loss + disc_real_loss) / 2
                #disc_loss.backward(retain_graph=True)
                disc_loss.backward()
                disc_opt.step()

            # Update generator
            # Zero out the generator gradients

            #here compute separate partial gradients (PnL, MSE, SR, STD, BCE) and measure each one’s norm. This is the “gradient check.”
            
            # generate another fake and pass to D
            noise = torch.randn(1,curr_batch_size, z_dim, device=device,dtype=torch.float)
            fake = gen(noise,condition,h_0g,c_0g)
            
            #fake1 = fake1.unsqueeze(0).unsqueeze(2)
            fake_and_condition = combine_vectors(condition,fake,dim=-1)

            disc_fake_pred = disc(fake_and_condition,h_0d,c_0d)
            
            # compute different partial “objectives” from the generator’s perspective:
            ft = fake.squeeze(0).squeeze(1)
            rl = real.squeeze(0).squeeze(1)
            
            sign_approx = torch.tanh(tanh_coeff * ft)
            PnL_s  = sign_approx * rl
            PnL = torch.mean(PnL_s)   # avg sign-based retursn
            MSE = (torch.norm(ft-rl)**2) / curr_batch_size  # difference between the generator’s forecast and real returns.
            SR = (torch.mean(PnL_s)) / (torch.std(PnL_s))   # Sharpe ratio (mean / std).
            STD = torch.std(PnL_s)   # STD: standard deviation of sign-based returns.
            gen_opt.zero_grad() 

            # one backward call for each of these partial losses, one at a time, measuring G's gradient noem
            SR.backward(retain_graph=True)
            total_norm = 0
            for p in gen.parameters():
                param_norm = p.grad.detach().data.norm(2)
                total_norm += param_norm.item() ** 2
            total_norm = total_norm ** (1. / 2)
            #list of gradient norms
            # measure gradient norm
            SR_norm[epoch*nbatches+i] = total_norm
            
            gen_opt.zero_grad() 
            PnL.backward(retain_graph = True)
            total_norm = 0
            for p in gen.parameters():
                param_norm = p.grad.detach().data.norm(2)
                total_norm += param_norm.item() ** 2
                total_norm = total_norm ** 0.5
            PnL_norm[epoch*nbatches+i] = total_norm
            
            gen_opt.zero_grad() 
            MSE.backward(retain_graph = True)
            total_norm = 0
            for p in gen.parameters():
                param_norm = p.grad.detach().data.norm(2)
                total_norm += param_norm.item() ** 2
                total_norm = total_norm ** 0.5
            MSE_norm[epoch*nbatches+i] = total_norm
            
            gen_opt.zero_grad() 
            STD.backward(retain_graph = True)
            total_norm = 0
            for p in gen.parameters():
                param_norm = p.grad.detach().data.norm(2)
                total_norm += param_norm.item() ** 2
                total_norm = total_norm ** 0.5
            STD_norm[epoch*nbatches+i] = total_norm
            # not updating G with these partial losses, just measure gradients, to see how these losses would shape the G's parameters

            # gen update
            gen_opt.zero_grad()   
            gen_loss = criterion(disc_fake_pred, torch.ones_like(disc_fake_pred))
            gen_loss.backward()
            total_norm = 0
            for p in gen.parameters():
                param_norm = p.grad.detach().data.norm(2)
                total_norm += param_norm.item() ** 2
                total_norm = total_norm ** 0.5
            BCE_norm[epoch*nbatches+i] = total_norm
            gen_opt.step()
            
# compute avg ratio of the BCE gradient norm to each partial gradient norm
# if alpha large, BCE gradients are on avg much bigger than PnL gradients -> poss. model more focused on fooling the D than improving PnL
# if alpha small, PnL gradients (e.g.) > BCE loss
    alpha = torch.mean(BCE_norm / PnL_norm)
    beta =  torch.mean(BCE_norm / MSE_norm)
    gamma =  torch.mean(BCE_norm / SR_norm)
    delta = torch.mean(BCE_norm / STD_norm)
    print("Completed. ")
    print(r"$\alpha$:", alpha)
    print(r"$\beta$:", beta)
    print(r"$\gamma$:", gamma)
    print(r"$\delta$:", delta)
    
    # plot the 5 gradient norm arrays if plot=True
    # if plot:
    #     plt.figure(ticker + " BCE norm")
    #     plt.title(ticker + " BCE norm")
    #     plt.plot(range(len(BCE_norm)),BCE_norm)
    #     plt.xlabel("iteration")
    #     plt.ylabel(r"$L^2$ norm")
    #     plt.show()
    
    #     plt.figure(ticker + " PnL norm")
    #     plt.title(ticker +" PnL norm")
    #     plt.plot(range(len(BCE_norm)),PnL_norm)
    #     plt.xlabel("iteration")
    #     plt.ylabel(r"$L^2$ norm")
    #     plt.show()
        
    #     plt.figure(ticker + " MSE norm")
    #     plt.title(ticker + " MSE norm")
    #     plt.plot(range(len(BCE_norm)), MSE_norm)
    #     plt.xlabel("iteration")
    #     plt.ylabel(r"$L^2$ norm")
    #     plt.show()
        
    #     plt.figure(ticker + " SR norm")
    #     plt.title("SR norm")
    #     plt.plot(range(len(BCE_norm)),SR_norm)
    #     plt.xlabel("iteration")
    #     plt.ylabel(r"$L^2$ norm")
    #     plt.show()
        
    #     plt.figure(ticker + " STD norm")
    #     plt.title(ticker + " STD norm")
    #     plt.plot(range(len(BCE_norm)),STD_norm)
    #     plt.ylabel(r"$L^2$ norm")
    #     plt.xlabel("iteration")
    #     plt.show()

    # CHANGED with
    if plot:
        plot_dir = os.path.join("Fin-GAN", "sector_results", "gradient_check_plots")
        os.makedirs(plot_dir, exist_ok=True)

        def save_plot(values, name):
            plt.figure(figsize=(8, 4))
            plt.plot(range(len(values)), values.cpu().numpy())
            plt.title(f"{ticker} — {name} norm")
            plt.xlabel("Iteration")
            plt.ylabel(r"$L^2$ norm")
            plt.grid(True)
            path = os.path.join(plot_dir, f"{ticker}_{name}_norm.png")
            plt.tight_layout()
            plt.savefig(path)
            plt.close()

        save_plot(BCE_norm, "BCE")
        save_plot(PnL_norm, "PnL")
        save_plot(MSE_norm, "MSE")
        save_plot(SR_norm, "SR")
        save_plot(STD_norm, "STD")
 
        # plt.figure(ticker + " Norms")
        # plt.title(ticker + " gradient norms")
        # plt.plot(range(len(BCE_norm)),BCE_norm, label = "BCE")
        # plt.plot(range(len(BCE_norm)),PnL_norm, label = "PnL")
        # plt.plot(range(len(BCE_norm)),SR_norm, label = "SR")
        # plt.plot(range(len(BCE_norm)),STD_norm, label = "STD")
        # plt.ylabel(r"$L^2$ norm")
        # plt.xlabel("iteration")
        # plt.legend(loc = 'best')
        # plt.show()
    
    # last four are average ratios of BCE gradient norm vs. other losses.
    return gen, disc, gen_opt, disc_opt, alpha, beta, gamma, delta

In [28]:
# Functions imported from FinGAN (Fin-GAN-online.py)
from FinGAN import rawreturns, combine_vectors

from FinGAN import (
    TrainLoopnLSTM, TrainLoopnLSTMSTD, TrainLoopnLSTMSR,
    TrainLoopnLSTMPnL, TrainLoopnLSTMPnLSR, TrainLoopnLSTMPnLSTD,
    Evaluation2LSTM
)

from FinGAN import (
    TrainLoopForGAN,
    TrainLoopMainPnLnv,
    TrainLoopMainPnLMSEnv,
    TrainLoopMainPnLMSESRnv,
    TrainLoopMainPnLMSESTDnv,
    TrainLoopMainMSEnv,
    TrainLoopMainSRnv,
    TrainLoopMainSRMSEnv,
    TrainLoopMainPnLSTDnv,
    Evaluation2,
    Generator,
    Discriminator
    #GradientCheck
)

#### Sectors

In [ ]:
sector_list = ['Information Technology','Consumer Discretionary','Financials','Health Care','Energy']

In [37]:
# WITH GRADIENT CHECK FLAG

import os
import pandas as pd
import torch
import io
import contextlib
from tqdm import tqdm
from itertools import product

# === Configuration ===
dataloc = "datasets/"
etflistloc = "datasets/stocks-etfs-list.csv"
ticker_meta = pd.read_csv(etflistloc)

sector_filter = "Energy"
filtered_tickers = ticker_meta[ticker_meta["Sector"] == sector_filter]["ticker_x"].tolist()

results_dir_base = f"./Fin-GAN/sector_results/{sector_filter}"
os.makedirs(results_dir_base, exist_ok=True)
hparam_log_path = os.path.join(results_dir_base, "gradient_hparams_log.csv")

# === Common Parameters ===
freq = 252
h, l, pred = 1, 10, 1
z_dim = 8
hid_d, hid_g = 64, 8
batch_size = 100
n_epochs = 20
lr_g, lr_d = 0.0001, 0.0001
device = "cpu"
use_gradient_check = True  #  GradientCheck flag for GAN

# === LSTM Loops ===
lstm_train_loops = [
    ("LSTM", TrainLoopnLSTM),
    ("LSTM_STD", TrainLoopnLSTMSTD),
    ("LSTM_SR", TrainLoopnLSTMSR),
    ("LSTM_PnL", TrainLoopnLSTMPnL),
    ("LSTM_PnL_SR", TrainLoopnLSTMPnLSR),
    ("LSTM_PnL_STD", TrainLoopnLSTMPnLSTD)
]

lstm_all_evals = []
total_tasks = list(product(filtered_tickers, lstm_train_loops))
last_ticker = None

for ticker, (tag, train_fn) in tqdm(total_tasks, desc="🖁️ Full LSTM Sector Training Progress"):
    try:
        if ticker != last_ticker:
            clear_output(wait=True)
            print(f"🔄 New Ticker: {ticker}")
            etf = ETF_find(etflistloc, ticker)
            if etf is None:
                print(f"⚠️ Skipping {ticker}")
                continue

            excess_ret, dates_dt = excessreturns(dataloc, ticker, etf)
            train_data, val_data, test_data, dates_used = split_from_returns(excess_ret, dates_dt, h=h, l=l, pred=pred)
            ref_mean = torch.mean(torch.tensor(train_data[:100, :]))
            ref_std = torch.std(torch.tensor(train_data[:100, :]))
            last_ticker = ticker

        gen = LSTM(noise_dim=0, cond_dim=l, hidden_dim=hid_g, output_dim=pred, mean=ref_mean, std=ref_std)
        gen_opt = torch.optim.RMSprop(gen.parameters(), lr=lr_g)

        with contextlib.redirect_stdout(io.StringIO()):
            gen, gen_opt = train_fn(
                gen=gen,
                gen_opt=gen_opt,
                criterion=False,
                alpha=1.0,
                beta=0.0,
                gamma=0.5,
                delta=0.1,
                n_epochs=n_epochs,
                checkpoint_epoch=10,
                train_data=torch.tensor(train_data, dtype=torch.float),
                validation_data=torch.tensor(val_data, dtype=torch.float),
                batch_size=batch_size,
                hid_d=hid_d,
                hid_g=hid_g,
                z_dim=z_dim,
                lr_d=lr_d,
                lr_g=lr_g,
                h=h,
                l=l,
                pred=pred,
                plot=False
            )

            df_eval, *_ = Evaluation2LSTM(
                ticker=ticker,
                freq=freq,
                gen=gen,
                test_data=torch.tensor(test_data, dtype=torch.float),
                val_data=torch.tensor(val_data, dtype=torch.float),
                h=h,
                l=l,
                pred=pred,
                hid_d=hid_d,
                hid_g=hid_g,
                z_dim=z_dim,
                lrg=lr_g,
                lrd=lr_d,
                n_epochs=n_epochs,
                losstype=tag,
                sr_val=0,
                device=device,
                plotsloc=results_dir_base,
                f_name=f"{ticker}_{tag}",
                plot=False
            )

        lstm_all_evals.append(df_eval)

    except Exception as e:
        print(f"❌ Error processing {ticker}-{tag}: {e}")

lstm_results = pd.concat(lstm_all_evals, ignore_index=True)
lstm_results.to_csv(os.path.join(results_dir_base, f"LSTM_summary_{sector_filter}.csv"), index=False)

# === GAN Loops ===
gan_train_loops = [
    ("GAN_BCE", TrainLoopForGAN),
    ("GAN_PnL", TrainLoopMainPnLnv),
    ("GAN_MSE", TrainLoopMainMSEnv),
    ("GAN_SR", TrainLoopMainSRnv),
    ("GAN_PnL_MSE", TrainLoopMainPnLMSEnv),
    ("GAN_PnL_SR", TrainLoopMainPnLMSESRnv),
    ("GAN_PnL_STD", TrainLoopMainPnLSTDnv),
    ("GAN_MSE_STD", TrainLoopMainPnLMSESTDnv),
    ("GAN_SR_MSE", TrainLoopMainSRMSEnv)
]

gan_all_evals = []
total_tasks = list(product(filtered_tickers, gan_train_loops))
last_ticker = None

for ticker, (tag, train_fn) in tqdm(total_tasks, desc="🖁️ Full GAN Sector Training Progress"):
    try:
        if ticker != last_ticker:
            clear_output(wait=True)
            print(f"🔄 New Ticker: {ticker}")
            etf = ETF_find(etflistloc, ticker)
            if etf is None:
                print(f"⚠️ Skipping {ticker}")
                continue

            excess_ret, dates_dt = excessreturns(dataloc, ticker, etf)
            train_data, val_data, test_data, dates_used = split_from_returns(excess_ret, dates_dt, h=h, l=l, pred=pred)
            ref_mean = torch.mean(torch.tensor(train_data[:100, :]))
            ref_std = torch.std(torch.tensor(train_data[:100, :]))
            last_ticker = ticker

        gen = Generator(z_dim, l, hid_g, pred, ref_mean, ref_std)
        disc = Discriminator(l + pred, hid_d, ref_mean, ref_std)
        gen_opt = torch.optim.RMSprop(gen.parameters(), lr=lr_g)
        disc_opt = torch.optim.RMSprop(disc.parameters(), lr=lr_d)
        criterion = torch.nn.BCELoss()

        # plot a few to have visual diagnostics of gradient behaviour over time, to be sure about num of epochs
        tickers_to_plot = filtered_tickers[:3]  # Pick the first 3 tickers from the sector 
        #tickers_to_plot = random.sample(filtered_tickers, k=3) #random

        if use_gradient_check:
            plot_flag = ticker in tickers_to_plot 
            gen, disc, gen_opt, disc_opt, alpha, beta, gamma, delta = GradientCheck(
                    ticker=ticker,
                    gen=gen,
                    disc=disc,
                    gen_opt=gen_opt,
                    disc_opt=disc_opt,
                    criterion=criterion,
                    n_epochs=5,  # gradient stabilize after around 3 epochs
                    train_data=torch.tensor(train_data, dtype=torch.float),
                    batch_size=batch_size,
                    hid_d=hid_d,
                    hid_g=hid_g,
                    z_dim=z_dim,
                    lr_d=lr_d,
                    lr_g=lr_g,
                    h=h,
                    l=l,
                    pred=pred,
                    device=device,
                    plot=plot_flag # plotting selectively
                )
            

            # Log best hparams
            pd.DataFrame([{
                "ticker": ticker,
                "tag": tag,
                "alpha": alpha,
                "beta": beta,
                "gamma": gamma,
                "delta": delta
            }]).to_csv(hparam_log_path, mode='a', index=False, header=not os.path.exists(hparam_log_path))
        else:
            alpha, beta, gamma, delta = 1.0, 0.0, 0.5, 0.1

        with contextlib.redirect_stdout(io.StringIO()):
            gen, disc, gen_opt, disc_opt = train_fn(
                gen=gen,
                disc=disc,
                gen_opt=gen_opt,
                disc_opt=disc_opt,
                criterion=criterion,
                alpha=alpha,
                beta=beta,
                gamma=gamma,
                delta=delta,
                n_epochs=n_epochs,
                checkpoint_epoch=10,
                train_data=torch.tensor(train_data, dtype=torch.float),
                validation_data=torch.tensor(val_data, dtype=torch.float),
                batch_size=batch_size,
                hid_d=hid_d,
                hid_g=hid_g,
                z_dim=z_dim,
                lr_d=lr_d,
                lr_g=lr_g,
                h=h,
                l=l,
                pred=pred,
                plot=False
            )

            df_eval, *_ = Evaluation2(
                ticker=ticker,
                freq=freq,
                gen=gen,
                test_data=torch.tensor(test_data, dtype=torch.float),
                val_data=torch.tensor(val_data, dtype=torch.float),
                h=h,
                l=l,
                pred=pred,
                hid_d=hid_d,
                hid_g=hid_g,
                z_dim=z_dim,
                lrg=lr_g,
                lrd=lr_d,
                n_epochs=n_epochs,
                losstype=tag,
                sr_val=0,
                device=device,
                plotsloc=results_dir_base,
                f_name=f"{ticker}_{tag}",
                plot=True
            )

        gan_all_evals.append(df_eval)

    except Exception as e:
        print(f"❌ Error processing {ticker}-{tag}: {e}")

gan_results = pd.concat(gan_all_evals, ignore_index=True)
gan_results.to_csv(os.path.join(results_dir_base, f"GAN_summary_{sector_filter}.csv"), index=False)

print(f"✅ Completed training for sector: {sector_filter}")


🔄 New Ticker: WMB
[ETF_find] Looking for ETF match for stock: WMB
[ETF_find] ✅ Found ETF XLE for stock WMB
[excessreturns] Length excess_ret: 7519, Length dates_double: 7519
[excessreturns] Sample s_ret:
[ 0.01071908 -0.00311483  0.01723029 -0.00703318  0.00878355]
[excessreturns] Sample e_ret:
[ 0.01002735  0.00227289  0.00157055 -0.00664802  0.00087726]
[excessreturns] Sample excess_ret:
[ 0.00069173 -0.00538772  0.01565975 -0.00038517  0.0079063 ]
[Train] Creating 6005 samples
[Train] Sample (first 2 rows):
[[ 0.00069173 -0.00538772  0.01565975 -0.00038517  0.0079063  -0.00149657
   0.01101495 -0.00118313  0.00302677 -0.00332635 -0.00586422]
 [-0.00538772  0.01565975 -0.00038517  0.0079063  -0.00149657  0.01101495
  -0.00118313  0.00302677 -0.00332635 -0.00586422  0.00553076]]
[Val] Creating 741 samples
[Val] Sample (first 2 rows):
[[-0.00480778  0.01390033  0.00395561 -0.00333503 -0.00291678 -0.00531052
  -0.00257663 -0.01416375 -0.00885858 -0.02303824  0.02875646]
 [ 0.01390033  0

100%|██████████| 5/5 [00:00<00:00,  8.34it/s]


Completed. 
$\alpha$: tensor(1.0044)
$\beta$: tensor(1.0331)
$\gamma$: tensor(1.7971)
$\delta$: tensor(0.9967)


100%|██████████| 5/5 [00:00<00:00,  8.43it/s]████████▌| 190/198 [06:37<00:20,  2.61s/it]


Completed. 
$\alpha$: tensor(1.0015)
$\beta$: tensor(1.0317)
$\gamma$: tensor(1.2148)
$\delta$: tensor(0.9942)


...

Completed. 
$\alpha$: tensor(1.0012)
$\beta$: tensor(1.0311)
$\gamma$: tensor(1.2921)
$\delta$: tensor(0.9934)


100%|██████████| 5/5 [00:00<00:00,  7.87it/s]████████▉| 197/198 [06:56<00:02,  2.68s/it]


Completed. 
$\alpha$: tensor(0.9986)
$\beta$: tensor(1.0248)
$\gamma$: tensor(2.4197)
$\delta$: tensor(0.9915)


🖁️ Full GAN Sector Training Progress: 100%|██████████| 198/198 [06:58<00:00,  2.12s/it]

✅ Completed training for sector: Energy


#### RESULTS ANALYSIS 
##### LSTM vs GAN comparison

In [38]:
# PER SECTOR
# === Config ===
sector_filter = "Energy"
base_dir = f"./Fin-GAN/sector_results/{sector_filter}"
lstm_path = os.path.join(base_dir, f"LSTM_summary_{sector_filter}.csv")
gan_path = os.path.join(base_dir, f"GAN_summary_{sector_filter}.csv")
plot_dir = os.path.join(base_dir, "plots")
os.makedirs(plot_dir, exist_ok=True)

# === Load Results ===
lstm_df = pd.read_csv(lstm_path)
gan_df = pd.read_csv(gan_path)

# === Add Model Label ===
lstm_df["Architecture"] = "LSTM"
gan_df["Architecture"] = "GAN"

# === Normalize Column Names for Join ===
lstm_df = lstm_df.rename(columns={
    "SR_m scaled test": "Sharpe",
    "PnL_m test": "PnL",
})
gan_df = gan_df.rename(columns={
    "SR_w scaled": "Sharpe",
    "PnL_w": "PnL",
})

# === Combine DataFrames ===
combined = pd.concat([lstm_df[["ticker", "type", "Sharpe", "PnL", "RMSE", "Architecture"]],
                      gan_df[["ticker", "type", "Sharpe", "PnL", "RMSE", "Architecture"]]],
                     ignore_index=True)

# === 📊 Plot: Sharpe Ratio Comparison ===
plt.figure(figsize=(10, 6))
sns.boxplot(data=combined, x="type", y="Sharpe", hue="Architecture")
plt.title(f"📈 Test Sharpe Ratio by Model — {sector_filter}")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(plot_dir, f"SR_comparison_{sector_filter}.png"))
plt.close()

# === 📊 Plot: RMSE Comparison ===
plt.figure(figsize=(10, 6))
sns.boxplot(data=combined, x="type", y="RMSE", hue="Architecture")
plt.title(f"📉 RMSE by Model — {sector_filter}")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(plot_dir, f"RMSE_comparison_{sector_filter}.png"))
plt.close()

# === 📋 Best-Model Summary by Ticker ===
def best_models_summary(df):
    return df.sort_values("Sharpe", ascending=False).groupby("ticker").first()

best_lstm = best_models_summary(lstm_df)
best_gan = best_models_summary(gan_df)

summary_df = pd.merge(best_lstm, best_gan, on="ticker", suffixes=("_LSTM", "_GAN"))
summary_df[["Sharpe_LSTM", "Sharpe_GAN", "PnL_LSTM", "PnL_GAN", "RMSE_LSTM", "RMSE_GAN"]].to_csv(
    os.path.join(base_dir, f"summary_best_comparison_{sector_filter}.csv")
)

# === Optional: Plot PnL Distributions ===
def plot_distributions(ticker):
    ticker_df = combined[combined["ticker"] == ticker]
    if not ticker_df.empty:
        plt.figure(figsize=(10, 4))
        sns.histplot(data=ticker_df, x="PnL", hue="Architecture", kde=True, bins=30)
        plt.title(f"{ticker} — Distribution of PnL")
        plt.tight_layout()
        plt.savefig(os.path.join(plot_dir, f"{ticker}_PnL_distribution.png"))
        plt.close()

# Plot top 5 tickers
for ticker in summary_df.index[:5]:
    plot_distributions(ticker)

print(f"✅ Analysis complete for sector: {sector_filter}")



/var/folders/k1/p_xw0_4x4zd812685rsx20380000gn/T/ipykernel_91292/2485578370.py:37: UserWarning: Glyph 128200 (\N{CHART WITH UPWARDS TREND}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/k1/p_xw0_4x4zd812685rsx20380000gn/T/ipykernel_91292/2485578370.py:38: UserWarning: Glyph 128200 (\N{CHART WITH UPWARDS TREND}) missing from font(s) DejaVu Sans.
  plt.savefig(os.path.join(plot_dir, f"SR_comparison_{sector_filter}.png"))
/var/folders/k1/p_xw0_4x4zd812685rsx20380000gn/T/ipykernel_91292/2485578370.py:46: UserWarning: Glyph 128201 (\N{CHART WITH DOWNWARDS TREND}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/k1/p_xw0_4x4zd812685rsx20380000gn/T/ipykernel_91292/2485578370.py:47: UserWarning: Glyph 128201 (\N{CHART WITH DOWNWARDS TREND}) missing from font(s) DejaVu Sans.
  plt.savefig(os.path.join(plot_dir, f"RMSE_comparison_{sector_filter}.png"))


✅ Analysis complete for sector: Energy


In [40]:

# COMPARISON BETWEEN SECTORS

# === Configuration ===
base_path = "./Fin-GAN/sector_results"
sectors = [
    "Information Technology",
    "Consumer Discretionary",
    "Financials",
    "Health Care",
    "Energy"
]

# === Containers ===
all_results = []

# === Load and Aggregate ===
for sector in sectors:
    try:
        lstm_path = os.path.join(base_path, sector, f"LSTM_summary_{sector}.csv")
        gan_path = os.path.join(base_path, sector, f"GAN_summary_{sector}.csv")

        lstm_df = pd.read_csv(lstm_path)
        gan_df = pd.read_csv(gan_path)

        # Standardize column names
        lstm_df = lstm_df.rename(columns={
            "SR_m scaled test": "SR_w scaled",
            "PnL_m test": "PnL_w"
        })

        lstm_df["Architecture"] = "LSTM"
        gan_df["Architecture"] = "GAN"
        lstm_df["Sector"] = sector
        gan_df["Sector"] = sector

        all_results.append(pd.concat([lstm_df, gan_df], ignore_index=True))

    except Exception as e:
        print(f"⚠️ Skipping {sector} due to error: {e}")

# === Combine All ===
combined_df = pd.concat(all_results, ignore_index=True)

# === Compute Sector-Level Means ===
sector_summary = combined_df.groupby(["Sector", "Architecture"]).agg({
    "SR_w scaled": "mean",
    "PnL_w": "mean",
    "RMSE": "mean"
}).reset_index()

# === Save Summary ===
summary_path = os.path.join(base_path, "sector_model_summary.csv")
sector_summary.to_csv(summary_path, index=False)
print(f"✅ Sector-level summary saved to: {summary_path}")

# === Plotting ===
plot_dir = os.path.join(base_path, "comparison_plots")
os.makedirs(plot_dir, exist_ok=True)

metrics = ["SR_w scaled", "PnL_w", "RMSE"]
for metric in metrics:
    plt.figure(figsize=(10, 6))
    sns.barplot(data=sector_summary, x="Sector", y=metric, hue="Architecture")
    plt.title(f"Average {metric} per Sector")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.grid(True, axis="y", linestyle="--", alpha=0.5)
    plt.savefig(os.path.join(plot_dir, f"{metric.replace(' ', '_')}_sector_comparison.png"))
    plt.close()

print("📊 Sector-wise comparison plots saved!")



✅ Sector-level summary saved to: ./Fin-GAN/sector_results/sector_model_summary.csv
📊 Sector-wise comparison plots saved!


In [41]:
#Compare model performance statistically across sectors
#Visualize Sharpe, PnL, and RMSE differences between GAN and LSTM
#Check which sectors show strongest relative gains

# === Load summary data ===
df = pd.read_csv("./Fin-GAN/sector_results/sector_model_summary.csv")

# === Pivot to Model Comparison Format ===
pivoted = df.pivot(index="Sector", columns="Architecture", values=["SR_w scaled", "PnL_w", "RMSE"])
pivoted.columns = ["_".join(col).strip() for col in pivoted.columns.values]
pivoted = pivoted.reset_index()

# === Compute Differences (GAN - LSTM) ===
pivoted["Sharpe_Diff"] = pivoted["SR_w scaled_GAN"] - pivoted["SR_w scaled_LSTM"]
pivoted["PnL_Diff"] = pivoted["PnL_w_GAN"] - pivoted["PnL_w_LSTM"]
pivoted["RMSE_Diff"] = pivoted["RMSE_LSTM"] - pivoted["RMSE_GAN"]  # lower is better

# === Plot Differences ===
metrics = ["Sharpe_Diff", "PnL_Diff", "RMSE_Diff"]
titles = ["Sharpe Ratio (GAN - LSTM)", "PnL (GAN - LSTM)", "RMSE (LSTM - GAN)"]

for metric, title in zip(metrics, titles):
    plt.figure(figsize=(10, 5))
    sns.barplot(data=pivoted, x="Sector", y=metric, palette="coolwarm", edgecolor="black")
    plt.title(f"Model Performance Difference by Sector: {title}")
    plt.axhline(0, color="black", linestyle="--")
    plt.xticks(rotation=45)
    plt.grid(True, axis="y", linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.savefig(f"./Fin-GAN/sector_results/perf_diff_{metric}.png")
    plt.close()

# === Optional: Display table
print("📋 Performance comparison (GAN - LSTM):")
print(pivoted[["Sector", "Sharpe_Diff", "PnL_Diff", "RMSE_Diff"]])




/var/folders/k1/p_xw0_4x4zd812685rsx20380000gn/T/ipykernel_91292/1791379287.py:26: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=pivoted, x="Sector", y=metric, palette="coolwarm", edgecolor="black")
/var/folders/k1/p_xw0_4x4zd812685rsx20380000gn/T/ipykernel_91292/1791379287.py:26: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=pivoted, x="Sector", y=metric, palette="coolwarm", edgecolor="black")
/var/folders/k1/p_xw0_4x4zd812685rsx20380000gn/T/ipykernel_91292/1791379287.py:26: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=pivot

📋 Performance comparison (GAN - LSTM):
                   Sector  Sharpe_Diff  PnL_Diff  RMSE_Diff
0  Consumer Discretionary     0.061237  0.247383   0.000112
1                  Energy     0.306160  0.231975   0.000132
2              Financials    -0.006516 -0.696012   0.000126
3             Health Care    -0.032877 -0.436725   0.000094
4  Information Technology     0.039731 -0.400958   0.000097


#### FINE TUNING
##### Rerun with finetuned hyperparameters

In [52]:
# tuned version

# === Config ===
sector_filter = "Financials"
etf_list_path = "datasets/stocks-etfs-list.csv"
dataloc = "datasets/"
ticker_meta = pd.read_csv(etf_list_path)
filtered_tickers = ticker_meta[ticker_meta["Sector"] == sector_filter]["ticker_x"].tolist()

output_base = f"./Fin-GAN/sector_results/{sector_filter}"
results_dir = os.path.join(output_base, "results_tuned")
plots_dir = os.path.join(output_base, "plots_tuned")
hparam_log_path = os.path.join(output_base, "gradient_hparams_log_tuned.csv")
os.makedirs(results_dir, exist_ok=True)
os.makedirs(plots_dir, exist_ok=True)

# === Parameters  ===
freq = 252
h, l, pred = 1, 10, 1
hid_d, hid_g, z_dim = 128, 16, 16
n_epochs = 30
batch_size = 100
lr_g, lr_d = 0.0002, 0.00005
alpha, beta, gamma, delta = 1.0, 0.2, 0.3, 0.1
device = "cpu"
use_gradient_check = True

# === LSTM Models to train ===
lstm_train_loops = [
    ("LSTM", TrainLoopnLSTM),
    ("LSTM_STD", TrainLoopnLSTMSTD),
    ("LSTM_SR", TrainLoopnLSTMSR),
    ("LSTM_PnL", TrainLoopnLSTMPnL),
    ("LSTM_PnL_SR", TrainLoopnLSTMPnLSR),
    ("LSTM_PnL_STD", TrainLoopnLSTMPnLSTD),
]

# === GAN Models to train ===
gan_train_loops = [
    ("GAN_BCE", TrainLoopForGAN),
    ("GAN_PnL", TrainLoopMainPnLnv),
    ("GAN_MSE", TrainLoopMainMSEnv),
    ("GAN_SR", TrainLoopMainSRnv),
    ("GAN_PnL_MSE", TrainLoopMainPnLMSEnv),
    ("GAN_PnL_SR", TrainLoopMainSRMSEnv),
    ("GAN_PnL_STD", TrainLoopMainPnLSTDnv),
    ("GAN_MSE_STD", TrainLoopMainPnLMSESTDnv),
    ("GAN_SR_MSE", TrainLoopMainSRMSEnv),
]

# === LSTM Training ===
lstm_all_evals = []
total_tasks = list(product(filtered_tickers, lstm_train_loops))
last_ticker = None

for ticker, (tag, train_fn) in tqdm(total_tasks, desc="📊 LSTM Fine-tuning"):
    try:
        if ticker != last_ticker:
            clear_output(wait=True)
            print(f"🔄 New Ticker: {ticker}")
            etf = ETF_find(etf_list_path, ticker)
            if etf is None:
                print(f"⚠️ Skipping {ticker}")
                continue

            excess_ret, dates_dt = excessreturns(dataloc, ticker, etf)
            train_data, val_data, test_data, _ = split_from_returns(excess_ret, dates_dt, h=h, l=l, pred=pred)
            ref_mean = torch.mean(torch.tensor(train_data[:100, :]))
            ref_std = torch.std(torch.tensor(train_data[:100, :]))
            last_ticker = ticker

        gen = LSTM(0, l, hid_g, pred, ref_mean, ref_std)
        gen_opt = torch.optim.RMSprop(gen.parameters(), lr=lr_g)

        with contextlib.redirect_stdout(io.StringIO()):
            gen, gen_opt = train_fn(
                gen, gen_opt, False, alpha, beta, gamma, delta, n_epochs, 10,
                torch.tensor(train_data, dtype=torch.float),
                torch.tensor(val_data, dtype=torch.float),
                batch_size, hid_d, hid_g, z_dim, lr_d, lr_g, h, l, pred, False
            )

            df_eval, *_ = Evaluation2LSTM(
                ticker, freq, gen,
                torch.tensor(test_data, dtype=torch.float),
                torch.tensor(val_data, dtype=torch.float),
                h, l, pred, hid_d, hid_g, z_dim, lr_g, lr_d, n_epochs,
                tag, 0, device, plots_dir, f"{ticker}_{tag}_tuned", False
            )

        lstm_all_evals.append(df_eval)

    except Exception as e:
        print(f"❌ LSTM Error for {ticker}-{tag}: {e}")

pd.concat(lstm_all_evals).to_csv(os.path.join(results_dir, f"LSTM_summary_{sector_filter}_tuned.csv"), index=False)

# === GAN Training ===
gan_all_evals = []
total_tasks = list(product(filtered_tickers, gan_train_loops))
last_ticker = None

for ticker, (tag, train_fn) in tqdm(total_tasks, desc="📊 GAN Fine-tuning"):
    try:
        if ticker != last_ticker:
            clear_output(wait=True)
            print(f"🔄 New Ticker: {ticker}")
            etf = ETF_find(etf_list_path, ticker)
            if etf is None:
                print(f"⚠️ Skipping {ticker}")
                continue

            excess_ret, dates_dt = excessreturns(dataloc, ticker, etf)
            train_data, val_data, test_data, dates_used = split_from_returns(excess_ret, dates_dt, h=h, l=l, pred=pred)
            ref_mean = torch.mean(torch.tensor(train_data[:100, :]))
            ref_std = torch.std(torch.tensor(train_data[:100, :]))
            last_ticker = ticker

        gen = Generator(z_dim, l, hid_g, pred, ref_mean, ref_std)
        disc = Discriminator(l + pred, hid_d, ref_mean, ref_std)
        gen_opt = torch.optim.RMSprop(gen.parameters(), lr=lr_g)
        disc_opt = torch.optim.RMSprop(disc.parameters(), lr=lr_d)
        criterion = torch.nn.BCELoss()

        tickers_to_plot = filtered_tickers[:3]

        if use_gradient_check:
            plot_flag = ticker in tickers_to_plot
            gen, disc, gen_opt, disc_opt, alpha, beta, gamma, delta = GradientCheck(
                ticker=ticker,
                gen=gen,
                disc=disc,
                gen_opt=gen_opt,
                disc_opt=disc_opt,
                criterion=criterion,
                n_epochs=5,
                train_data=torch.tensor(train_data, dtype=torch.float),
                batch_size=batch_size,
                hid_d=hid_d,
                hid_g=hid_g,
                z_dim=z_dim,
                lr_d=lr_d,
                lr_g=lr_g,
                h=h,
                l=l,
                pred=pred,
                device=device,
                plot=plot_flag
            )

            pd.DataFrame([{
                "ticker": ticker,
                "tag": tag,
                "alpha": alpha,
                "beta": beta,
                "gamma": gamma,
                "delta": delta
            }]).to_csv(hparam_log_path, mode='a', index=False, header=not os.path.exists(hparam_log_path))
        else:
            alpha, beta, gamma, delta = 1.0, 0.0, 0.5, 0.1

        with contextlib.redirect_stdout(io.StringIO()):
            gen, disc, gen_opt, disc_opt = train_fn(
                gen=gen,
                disc=disc,
                gen_opt=gen_opt,
                disc_opt=disc_opt,
                criterion=criterion,
                alpha=alpha,
                beta=beta,
                gamma=gamma,
                delta=delta,
                n_epochs=n_epochs, 
                checkpoint_epoch=10,
                train_data=torch.tensor(train_data, dtype=torch.float),
                validation_data=torch.tensor(val_data, dtype=torch.float),
                batch_size=batch_size, 
                hid_d=hid_d,
                hid_g=hid_g,
                z_dim=z_dim,
                lr_d=lr_d,
                lr_g=lr_g,
                h=h,
                l=l,
                pred=pred,
                plot=False
            )


            df_eval, *_ = Evaluation2(
                ticker=ticker,
                freq=freq,
                gen=gen,
                test_data=torch.tensor(test_data, dtype=torch.float),
                val_data=torch.tensor(val_data, dtype=torch.float),
                h=h,
                l=l,
                pred=pred,
                hid_d=hid_d,
                hid_g=hid_g,
                z_dim=z_dim,
                lrg=lr_g,
                lrd=lr_d,
                n_epochs=n_epochs,
                losstype=tag,
                sr_val=0,
                device=device,
                plotsloc=plots_dir,
                f_name=f"{ticker}_{tag}_tuned",
                plot=False
            )

        gan_all_evals.append(df_eval)

    except Exception as e:
        print(f"❌ GAN Error for {ticker}-{tag}: {e}")

pd.concat(gan_all_evals).to_csv(os.path.join(results_dir, f"GAN_summary_{sector_filter}_tuned.csv"), index=False)

print(f"✅ Finished training tuned models for sector: {sector_filter}")


🔄 New Ticker: WTW
[ETF_find] Looking for ETF match for stock: WTW
[ETF_find] ✅ Found ETF XLF for stock WTW
[excessreturns] Length excess_ret: 7519, Length dates_double: 7519
[excessreturns] Sample s_ret:
[ 0.00794036  0.00189634  0.00786285 -0.00282367 -0.00946975]
[excessreturns] Sample e_ret:
[ 0.01533576  0.00253308  0.00567648 -0.00630954  0.00693798]
[excessreturns] Sample excess_ret:
[-0.0073954  -0.00063675  0.00218638  0.00348587 -0.01640773]
[Train] Creating 6005 samples
[Train] Sample (first 2 rows):
[[-0.0073954  -0.00063675  0.00218638  0.00348587 -0.01640773 -0.00340783
   0.0100568  -0.00531229  0.00280609 -0.00087035  0.01189538]
 [-0.00063675  0.00218638  0.00348587 -0.01640773 -0.00340783  0.0100568
  -0.00531229  0.00280609 -0.00087035  0.01189538 -0.00156291]]
[Val] Creating 741 samples
[Val] Sample (first 2 rows):
[[-0.00262744 -0.000561    0.00119984  0.00579199  0.0088781  -0.00117344
   0.00045194  0.01589184 -0.0059205   0.01101078 -0.0120302 ]
 [-0.000561    0.

100%|██████████| 5/5 [00:00<00:00,  5.98it/s]


Completed. 
$\alpha$: tensor(0.9963)
$\beta$: tensor(1.0235)
$\gamma$: tensor(1.2558)
$\delta$: tensor(0.9903)


100%|██████████| 5/5 [00:00<00:00,  6.11it/s][53:29<00:42,  5.28s/it]


Completed. 
$\alpha$: tensor(1.0013)
$\beta$: tensor(1.0277)
$\gamma$: tensor(1.5056)
$\delta$: tensor(0.9953)


...

Completed. 
$\alpha$: tensor(0.9953)
$\beta$: tensor(1.0223)
$\gamma$: tensor(1.1368)
$\delta$: tensor(0.9895)


100%|██████████| 5/5 [00:00<00:00,  6.04it/s][54:06<00:05,  5.40s/it]


Completed. 
$\alpha$: tensor(1.0005)
$\beta$: tensor(1.0274)
$\gamma$: tensor(1.2124)
$\delta$: tensor(0.9948)


📊 GAN Fine-tuning: 100%|██████████| 657/657 [54:12<00:00,  4.95s/it]

✅ Finished training tuned models for sector: Financials


#### comparison between base and tuned (for single sector)

In [ ]:
# === Config ===
sector_filter = "Consumer Discretionary"
base_dir = f"./Fin-GAN/sector_results/{sector_filter}" # need to copy the tuned results in the sector_results folder
plot_dir = os.path.join(base_dir, "plots_tuned_vs_base")
os.makedirs(plot_dir, exist_ok=True)

# === Load Results ===
lstm_base = pd.read_csv(os.path.join(base_dir, f"LSTM_summary_{sector_filter}.csv"))
lstm_tuned = pd.read_csv(os.path.join(base_dir, f"LSTM_summary_{sector_filter}_tuned.csv"))
gan_base = pd.read_csv(os.path.join(base_dir, f"GAN_summary_{sector_filter}.csv"))
gan_tuned = pd.read_csv(os.path.join(base_dir, f"GAN_summary_{sector_filter}_tuned.csv"))

# === Normalize Columns ===
lstm_base = lstm_base.rename(columns={"SR_m scaled test": "Sharpe", "PnL_m test": "PnL"})
lstm_tuned = lstm_tuned.rename(columns={"SR_m scaled test": "Sharpe", "PnL_m test": "PnL"})
gan_base = gan_base.rename(columns={"SR_w scaled": "Sharpe", "PnL_w": "PnL"})
gan_tuned = gan_tuned.rename(columns={"SR_w scaled": "Sharpe", "PnL_w": "PnL"})

# === Add Labels ===
lstm_base["Architecture"] = "LSTM_base"
lstm_tuned["Architecture"] = "LSTM_tuned"
gan_base["Architecture"] = "GAN_base"
gan_tuned["Architecture"] = "GAN_tuned"

# === Combine All ===
combined = pd.concat([lstm_base, lstm_tuned, gan_base, gan_tuned], ignore_index=True)

# === Plot: Sharpe Ratio ===
plt.figure(figsize=(12, 6))
sns.boxplot(data=combined, x="type", y="Sharpe", hue="Architecture")
plt.title(f"Test Sharpe Ratio Comparison — {sector_filter}")
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(plot_dir, f"Sharpe_comparison_{sector_filter}.png"))
plt.close()

# === Plot: RMSE ===
plt.figure(figsize=(12, 6))
sns.boxplot(data=combined, x="type", y="RMSE", hue="Architecture")
plt.title(f"RMSE Comparison — {sector_filter}")
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(plot_dir, f"RMSE_comparison_{sector_filter}.png"))
plt.close()

# === Summary Table ===
def best_models_summary(df):
    return df.sort_values("Sharpe", ascending=False).groupby("ticker").first()

best_lstm_base = best_models_summary(lstm_base)
best_lstm_tuned = best_models_summary(lstm_tuned)
best_gan_base = best_models_summary(gan_base)
best_gan_tuned = best_models_summary(gan_tuned)

summary_df = pd.DataFrame({
    "Sharpe_LSTM_base": best_lstm_base["Sharpe"],
    "Sharpe_LSTM_tuned": best_lstm_tuned["Sharpe"],
    "Sharpe_GAN_base": best_gan_base["Sharpe"],
    "Sharpe_GAN_tuned": best_gan_tuned["Sharpe"],
    "RMSE_LSTM_base": best_lstm_base["RMSE"],
    "RMSE_LSTM_tuned": best_lstm_tuned["RMSE"],
    "RMSE_GAN_base": best_gan_base["RMSE"],
    "RMSE_GAN_tuned": best_gan_tuned["RMSE"],
})

summary_df.to_csv(os.path.join(base_dir, f"summary_finetune_comparison_{sector_filter}.csv"))

# === Plot: Average PnL Comparison ===
sns.set_theme(style="whitegrid")

# === Compute Mean PnL ===
avg_pnl_df = combined.groupby("Architecture")["PnL"].mean().reset_index()

# === Plot ===
plt.figure(figsize=(10, 6))
barplot = sns.barplot(
    data=avg_pnl_df,
    x="Architecture",
    y="PnL",
    palette="pastel",
    edgecolor=".6"
)

# === Add Bar Annotations ===
for i, row in avg_pnl_df.iterrows():
    barplot.text(
        i,
        row["PnL"] + 0.001,
        f"{row['PnL']:.4f}",
        ha='center',
        va='bottom',
        fontsize=11,
        fontweight='bold',
        color="#444"
    )

# === Customize Appearance ===
plt.title(f" Average PnL per Model — {sector_filter}", fontsize=18, fontweight='bold')
plt.ylabel("Average PnL", fontsize=14)
plt.xlabel("")
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.grid(axis="y", linestyle="--", alpha=0.6)
sns.despine(left=True, bottom=True)

plt.tight_layout()
plt.savefig(os.path.join(plot_dir, f"PnL_avg_comparison_{sector_filter}.png"))
plt.close()



print(f" Finished tuned vs base analysis for: {sector_filter}")

# outputs is folder plots_tuned_vs_base


#### FINAL COMPARISON BASE VS TUNED ACROSS SECTORS

In [ ]:
# === CONFIGURATION ===
base_dir = "./Fin-GAN/sector_results"
sectors = ["Information Technology", "Energy", "Health Care", "Financials", "Consumer Discretionary"]
metrics = ["PnL", "Sharpe", "RMSE"]
plot_labels = {
    "PnL": "Average PnL",
    "Sharpe": "Average Sharpe Ratio",
    "RMSE": "Average RMSE"
}
architecture_order = ["LSTM_base", "LSTM_tuned", "GAN_base", "GAN_tuned"]
colors = {
    "LSTM_base": "#a6cee3",
    "LSTM_tuned": "#1f78b4",
    "GAN_base": "#b2df8a",
    "GAN_tuned": "#33a02c"
}
output_dir = os.path.join(base_dir, "comparison_plots_across_sectors")
os.makedirs(output_dir, exist_ok=True)

# === COLLECT AND PREPARE DATA ===
summary_rows = []

for sector in sectors:
    try:
        lstm_base = pd.read_csv(os.path.join(base_dir, sector, f"LSTM_summary_{sector}.csv"))
        lstm_tuned = pd.read_csv(os.path.join(base_dir, sector, f"LSTM_summary_{sector}_tuned.csv"))
        gan_base = pd.read_csv(os.path.join(base_dir, sector, f"GAN_summary_{sector}.csv"))
        gan_tuned = pd.read_csv(os.path.join(base_dir, sector, f"GAN_summary_{sector}_tuned.csv"))

        # Normalize columns
        for df in [lstm_base, lstm_tuned]:
            df.rename(columns={"SR_m scaled test": "Sharpe", "PnL_m test": "PnL"}, inplace=True)
        for df in [gan_base, gan_tuned]:
            df.rename(columns={"SR_w scaled": "Sharpe", "PnL_w": "PnL"}, inplace=True)

        # Label and merge
        lstm_base["Architecture"] = "LSTM_base"
        lstm_tuned["Architecture"] = "LSTM_tuned"
        gan_base["Architecture"] = "GAN_base"
        gan_tuned["Architecture"] = "GAN_tuned"
        combined = pd.concat([lstm_base, lstm_tuned, gan_base, gan_tuned], ignore_index=True)
        combined["Sector"] = sector

        # Group and average
        avg = combined.groupby("Architecture").agg({
            "PnL": "mean",
            "Sharpe": "mean",
            "RMSE": "mean"
        }).reset_index()
        avg["Sector"] = sector
        summary_rows.append(avg)

    except Exception as e:
        print(f" Skipping {sector} due to error: {e}")

# === COMBINE + PREP FOR PLOTTING ===
summary_df = pd.concat(summary_rows, ignore_index=True)
summary_df["Architecture"] = pd.Categorical(summary_df["Architecture"], categories=architecture_order, ordered=True)
summary_df.sort_values(["Sector", "Architecture"], inplace=True)

# === PLOT WITH BEST MODEL HIGHLIGHT AND RANKS ===
for metric in metrics:
    plt.figure(figsize=(12, 6))
    ax = sns.barplot(
        data=summary_df,
        x="Sector",
        y=metric,
        hue="Architecture",
        order=sectors,
        hue_order=architecture_order,
        palette=colors
    )

    # Highlight best bar per sector
    for i, sector in enumerate(sectors):
        subset = summary_df[summary_df["Sector"] == sector]
        if metric == "RMSE":
            best_idx = subset[metric].idxmin()
        else:
            best_idx = subset[metric].idxmax()

        best_model = summary_df.loc[best_idx, "Architecture"]
        best_val = summary_df.loc[best_idx, metric]
        ax.text(i, best_val + 0.02, "★", ha="center", va="bottom", fontsize=14, color="gold")

        # Add ranks above bars
        ranked = subset.sort_values(by=metric, ascending=(metric == "RMSE"))
        for rank, (_, row) in enumerate(ranked.iterrows(), start=1):
            arch = row["Architecture"]
            val = row[metric]
            xpos = i - 0.3 + architecture_order.index(arch) * 0.2
            ax.text(xpos, val + 0.01, f"{rank}", ha="center", fontsize=9, color="black")

    plt.title(f"{plot_labels[metric]} Across Sectors", fontsize=14, weight="bold")
    plt.ylabel(plot_labels[metric])
    plt.xticks(rotation=45)
    plt.grid(True, axis="y", linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.legend(title="Model", loc="upper right")
    plt.savefig(os.path.join(output_dir, f"{metric.lower()}_comparison_across_sectors.png"))
    plt.close()

print("sector-level comparison plots saved!")


In [72]:
# ALL SECTORS
# statistical comparison between base and tuned models (LSTM vs GAN) across sectors
# Load the summary.
# - Run paired t-tests comparing base vs tuned versions (per model type) for both Sharpe and RMSE.
# - Show statistical significance (p-values).
# - Output whether tuning led to a significant improvement.

# === Configuration ===
sectors = ["Information Technology", "Energy", "Financials", "Health Care", "Consumer Discretionary"]
base_path = "./Fin-GAN/sector_results"

# === Metrics to Compare ===
comparisons = [
    ("Sharpe Ratio (LSTM)", "Sharpe_LSTM_base", "Sharpe_LSTM_tuned"),
    ("Sharpe Ratio (GAN)", "Sharpe_GAN_base", "Sharpe_GAN_tuned"),
    ("RMSE (LSTM)", "RMSE_LSTM_base", "RMSE_LSTM_tuned"),
    ("RMSE (GAN)", "RMSE_GAN_base", "RMSE_GAN_tuned")
]

# === Loop through sectors and run t-tests ===
for sector in sectors:
    print(f"\n📊 Sector: {sector}")
    path = os.path.join(base_path, sector, f"summary_finetune_comparison_{sector}.csv")

    if not os.path.exists(path):
        print("⚠️ Missing summary file. Skipping...")
        continue

    df = pd.read_csv(path).dropna()

    for label, base_col, tuned_col in comparisons:
        base = df[base_col]
        tuned = df[tuned_col]
        t_stat, p_val = ttest_rel(tuned, base)

        print(f"\n🔎 {label}")
        print(f"→ t-statistic = {t_stat:.4f}, p-value = {p_val:.4f}")
        print(f"→ Mean (Base): {base.mean():.4f} | Mean (Tuned): {tuned.mean():.4f}")

        if p_val < 0.05:
            if tuned.mean() > base.mean():
                print("✅ Tuned version is significantly better.")
            else:
                print("🔻 Tuned version is significantly worse.")
        else:
            print("⚠️ No significant difference.")



📊 Sector: Information Technology

🔎 Sharpe Ratio (LSTM)
→ t-statistic = 1.6146, p-value = 0.1122
→ Mean (Base): 0.8858 | Mean (Tuned): 1.0436
⚠️ No significant difference.

🔎 Sharpe Ratio (GAN)
→ t-statistic = 1.7288, p-value = 0.0896
→ Mean (Base): 0.9518 | Mean (Tuned): 1.0822
⚠️ No significant difference.

🔎 RMSE (LSTM)
→ t-statistic = -0.7728, p-value = 0.4430
→ Mean (Base): 0.0133 | Mean (Tuned): 0.0133
⚠️ No significant difference.

🔎 RMSE (GAN)
→ t-statistic = 3.8297, p-value = 0.0003
→ Mean (Base): 0.0133 | Mean (Tuned): 0.0134
✅ Tuned version is significantly better.

📊 Sector: Energy

🔎 Sharpe Ratio (LSTM)
→ t-statistic = 1.2704, p-value = 0.2221
→ Mean (Base): 0.9977 | Mean (Tuned): 1.2530
⚠️ No significant difference.

🔎 Sharpe Ratio (GAN)
→ t-statistic = 0.0288, p-value = 0.9774
→ Mean (Base): 1.3650 | Mean (Tuned): 1.3703
⚠️ No significant difference.

🔎 RMSE (LSTM)
→ t-statistic = -1.5204, p-value = 0.1479
→ Mean (Base): 0.0082 | Mean (Tuned): 0.0081
⚠️ No significant d

#### COMPARISON WITH ARIMA 

In [85]:
# loop through tickers per sector and collect ARIMA evaluation results 


# === Parameters ===
dataloc = "datasets/"
etf_list_path = "datasets/stocks-etfs-list.csv"
ticker_meta = pd.read_csv(etf_list_path)

h, l, pred = 1, 10, 1

# === Function to Evaluate Fixed ARIMA(p=1,d=0,q=0) ===
def evaluate_simple_arima(train_data, test_data):
    try:
        model = ARIMA(train_data, order=(1, 0, 0))
        model_fit = model.fit()
        forecast = model_fit.forecast(steps=len(test_data))

        rmse = np.sqrt(mean_squared_error(test_data, forecast))
        pnl = np.sum(np.sign(forecast) * test_data)
        sharpe = np.mean(np.sign(forecast) * test_data) / (np.std(np.sign(forecast) * test_data) + 1e-6)

        return rmse, pnl, sharpe
    except Exception as e:
        return None, None, None

# === ARIMA Evaluation for a Single Ticker ===
def evaluate_ticker_arima(ticker, sector):
    try:
        etf = ETF_find(etf_list_path, ticker)
        if etf is None:
            return None

        excess_ret, dates_dt = excessreturns(dataloc, ticker, etf)
        train_data, val_data, test_data, _ = split_from_returns(excess_ret, dates_dt, h=h, l=l, pred=pred)
        full_train = np.concatenate([train_data.flatten(), val_data.flatten()])[-1000:]  # Limit history

        rmse, pnl, sharpe = evaluate_simple_arima(full_train, test_data[:, -1])

        if rmse is None:
            return None

        return {
            "ticker": ticker,
            "Sector": sector,
            "Sharpe": sharpe,
            "RMSE": rmse,
            "PnL": pnl,
            "Model": "ARIMA"
        }
    except Exception as e:
        return None

# === Loop Through Specific Sectors Only ===
all_sector_results = []
sectors_considered = [
    "Information Technology",
    "Energy",
    "Financials",
    "Health Care",
    "Consumer Discretionary"
]

for sector in sectors_considered:
    clear_output(wait=True)
    print(f"\n🔍 Evaluating ARIMA for sector: {sector}")
    filtered_tickers = ticker_meta[ticker_meta["Sector"] == sector]["ticker_x"].tolist()[:30]  # Limit to 30 tickers

    sector_results = Parallel(n_jobs=4)(
        delayed(evaluate_ticker_arima)(ticker, sector) for ticker in filtered_tickers
    )

    sector_results = [res for res in sector_results if res is not None]
    all_sector_results.extend(sector_results)

# === Convert to DataFrame ===
arima_df = pd.DataFrame(all_sector_results)

# === Save Results ===
arima_df.to_csv("./Fin-GAN/arima_sector_results.csv", index=False)

# === Plot Boxplots Across Sectors ===
metrics = ["Sharpe", "PnL", "RMSE"]

os.makedirs("./Fin-GAN/plots", exist_ok=True)

for metric in metrics:
    plt.figure(figsize=(12, 6))
    sns.boxplot(data=arima_df, x="Sector", y=metric, color="skyblue")
    plt.title(f"ARIMA Model — {metric} Across Sectors")
    plt.xticks(rotation=45)
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(f"./Fin-GAN/plots_arima/arima_{metric.lower()}_across_sectors.png")
    plt.close()

print("ARIMA sector evaluation complete and visualized.")




🔍 Evaluating ARIMA for sector: Consumer Discretionary
[ETF_find] Looking for ETF match for stock: ABNB
[ETF_find] ✅ Found ETF XLY for stock ABNB
[ETF_find] Looking for ETF match for stock: AMZN
[ETF_find] ✅ Found ETF XLY for stock AMZN
[ETF_find] Looking for ETF match for stock: APTV
[ETF_find] ✅ Found ETF XLY for stock APTV
[ETF_find] Looking for ETF match for stock: AZO
[ETF_find] ✅ Found ETF XLY for stock AZO
[ETF_find] Looking for ETF match for stock: BBY
[ETF_find] Looking for ETF match for stock: BKNG
[ETF_find] ✅ Found ETF XLY for stock BBY
[ETF_find] ✅ Found ETF XLY for stock BKNG
[excessreturns] Length excess_ret: 7519, Length dates_double: 7519
[excessreturns] Sample s_ret:
[ 0.00888028 -0.00218394  0.00710893 -0.0041216  -0.00436905]
[excessreturns] Sample e_ret:
[ 0.01974164  0.00274523  0.00121757 -0.00549124  0.00152844]
[excessreturns] Sample excess_ret:
[-0.01086136 -0.00492916  0.00589136  0.00136964 -0.00589749]
[excessreturns] Length excess_ret: 7519, Length dates_d

/Users/federicadani/miniconda3/envs/MLFinance/lib/python3.10/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/federicadani/miniconda3/envs/MLFinance/lib/python3.10/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


[ETF_find] ✅ Found ETF XLY for stock BWA
[ETF_find] Looking for ETF match for stock: CZR
[ETF_find] ✅ Found ETF XLY for stock CZR
[ETF_find] Looking for ETF match for stock: KMX
[ETF_find] Looking for ETF match for stock: CMG
[ETF_find] ✅ Found ETF XLY for stock KMX
[ETF_find] ✅ Found ETF XLY for stock CMG
[ETF_find] Looking for ETF match for stock: DECK
[ETF_find] ✅ Found ETF XLY for stock DECK
[excessreturns] Length excess_ret: 7519, Length dates_double: 7519
[excessreturns] Sample s_ret:
[ 0.04392459 -0.00263994  0.00448369 -0.00607078  0.00211575]
[excessreturns] Sample e_ret:
[ 0.01974164  0.00274523  0.00121757 -0.00549124  0.00152844]
[excessreturns] Sample excess_ret:
[ 0.02418295 -0.00538517  0.00326612 -0.00057954  0.0005873 ]
[Train] Creating 6005 samples
[Train] Sample (first 2 rows):
[[ 0.02418295 -0.00538517  0.00326612 -0.00057954  0.0005873   0.00163896
  -0.01965328 -0.0009825  -0.0126326  -0.00297085  0.00253917]
 [-0.00538517  0.00326612 -0.00057954  0.0005873   0.00

/Users/federicadani/miniconda3/envs/MLFinance/lib/python3.10/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


✅ ARIMA sector evaluation complete and visualized.


#### Average metrics computation for results sharing

In [103]:
# AVERAGE RESULTS COMPUTATION FOR COMPARISON

for metric in metrics:
    plt.figure(figsize=(12, 6))
    
    # Compute average per sector
    sector_avg = arima_df.groupby("Sector")[metric].mean().reset_index()
    
    # Plot bars
    ax = sns.barplot(data=sector_avg, x="Sector", y=metric, color="skyblue")
    
    # Annotate value inside each bar
    for i, row in sector_avg.iterrows():
        ax.text(
            i, row[metric] / 2,  
            f"{row[metric]:.4f}",
            ha="center", va="center", fontsize=9, color="black", weight="bold"
        )
    
    plt.title(f"ARIMA Model — {metric} Across Sectors", fontsize=14, weight="bold")
    plt.ylabel(metric)
    plt.xticks(rotation=45)
    plt.grid(True, axis="y", linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.savefig(f"./Fin-GAN/plots_arima/arima_{metric.lower()}_across_sectors_bars.png")
    plt.close()



In [ ]:
#################################################

# === CONFIGURATION ===
base_dir = "./Fin-GAN/sector_results"
sectors = ["Information Technology", "Energy", "Health Care", "Financials", "Consumer Discretionary"]
metrics = ["PnL", "Sharpe", "RMSE"]
plot_labels = {
    "PnL": "Average PnL",
    "Sharpe": "Average Sharpe Ratio",
    "RMSE": "Average RMSE"
}
architecture_order = ["LSTM_base", "LSTM_tuned", "GAN_base", "GAN_tuned"]
colors = {
    "LSTM_base": "#a6cee3",
    "LSTM_tuned": "#1f78b4",
    "GAN_base": "#b2df8a",
    "GAN_tuned": "#33a02c"
}
output_dir = os.path.join(base_dir, "comparison_plots_across_sectors")
os.makedirs(output_dir, exist_ok=True)

summary_rows = []

# === COLLECT AND PREPARE DATA ===
for sector in sectors:
    try:
        lstm_base = pd.read_csv(os.path.join(base_dir, sector, f"LSTM_summary_{sector}.csv"))
        lstm_tuned = pd.read_csv(os.path.join(base_dir, sector, f"LSTM_summary_{sector}_tuned.csv"))
        gan_base = pd.read_csv(os.path.join(base_dir, sector, f"GAN_summary_{sector}.csv"))
        gan_tuned = pd.read_csv(os.path.join(base_dir, sector, f"GAN_summary_{sector}_tuned.csv"))

        # Normalize column names
        for df in [lstm_base, lstm_tuned]:
            df.rename(columns={"SR_m scaled test": "Sharpe", "PnL_m test": "PnL"}, inplace=True)
        for df in [gan_base, gan_tuned]:
            df.rename(columns={"SR_w scaled": "Sharpe", "PnL_w": "PnL"}, inplace=True)

        # Label models
        lstm_base["Architecture"] = "LSTM_base"
        lstm_tuned["Architecture"] = "LSTM_tuned"
        gan_base["Architecture"] = "GAN_base"
        gan_tuned["Architecture"] = "GAN_tuned"

        combined = pd.concat([lstm_base, lstm_tuned, gan_base, gan_tuned], ignore_index=True)
        combined["Sector"] = sector

        # Group by architecture and compute averages per metric
        avg = combined.groupby("Architecture")[["PnL", "Sharpe", "RMSE"]].mean().reset_index()
        avg["Sector"] = sector
        summary_rows.append(avg)

    except Exception as e:
        print(f"❌ Skipping {sector}: {e}")

# === AGGREGATE AND PLOT ===
summary_df = pd.concat(summary_rows, ignore_index=True)
summary_df["Architecture"] = pd.Categorical(summary_df["Architecture"], categories=architecture_order, ordered=True)
summary_df.sort_values(["Sector", "Architecture"], inplace=True)

# Compute overall averages
overall_avg = summary_df.groupby("Architecture")[["PnL", "Sharpe", "RMSE"]].mean().round(4)
print("🔍 Overall averages:\n", overall_avg)

# === PLOT RESULTS PER METRIC ===
for metric in metrics:
    plt.figure(figsize=(18, 10))
    ax = sns.barplot(
        data=summary_df,
        x="Sector",
        y=metric,
        hue="Architecture",
        order=sectors,
        hue_order=architecture_order,
        palette=colors
    )

    # Annotate best model + rank + value
    for i, sector in enumerate(sectors):
        subset = summary_df[summary_df["Sector"] == sector]
        best_idx = subset[metric].idxmin() if metric == "RMSE" else subset[metric].idxmax()
        best_val = summary_df.loc[best_idx, metric]
        ax.text(i, best_val + 0.02,'★', ha="center", va="bottom", fontsize=14, color="gold")

        # Annotate rank and value above bars
        ranked = subset.sort_values(by=metric, ascending=(metric == "RMSE"))
        offset = 0.002 if metric == "RMSE" else 0.01
        # for rank, (_, row) in enumerate(ranked.iterrows(), start=1):
        #     arch = row["Architecture"]
        #     val = row[metric]
        #     xpos = i - 0.3 + architecture_order.index(arch) * 0.2
            
        #     ax.text(xpos, val + offset, f"{rank}\n{val:.2f}", ha="center", fontsize=8)

        #     #ax.text(xpos, val + 0.01, f"{rank}\n{val:.2f}", ha="center", fontsize=8)

        # Set vertical position inside the bar
        for rank, (_, row) in enumerate(ranked.iterrows(), start=1):
            arch = row["Architecture"]
            val = row[metric]
            xpos = i - 0.3 + architecture_order.index(arch) * 0.2

            # Show metric value inside the bar
            ax.text(
                xpos,
                val / 2,  # Center vertically inside the bar
                f"{val:.4f}",
                ha="center",
                va="center",
                fontsize=8,
                color="black" if val < 0.2 else "white",  # contrast text color
                fontweight="bold"
            )


    plt.title(f"{plot_labels[metric]} Across Sectors", fontsize=14, weight="bold")
    plt.ylabel(plot_labels[metric])
    #plt.xticks(rotation=45)
    plt.xticks(rotation=30, ha="right")
    plt.grid(True, axis="y", linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.legend(title="Model", loc="upper right")
    plt.savefig(os.path.join(output_dir, f"{metric.lower()}_comparison_across_sectors.png"))
    plt.close()

print("Comparison plots saved with average values and rankings.")
